# Data Leakage

[[cross-validation]] gives you an honest estimate — *provided the folds are
honest*. Leakage is what makes them dishonest.

A leak is any path by which information from outside a model's legitimate
training data reaches it. The symptom is always the same and always seductive:
**validation scores that are too good, and a model that disappoints in
production.** Nothing errors. Nothing warns you. The number simply is not
measuring what you think it is.

This page works through five kinds of leak. Each one is built deliberately,
measured against the honest baseline, and then fixed — so the size of the
optimism is a number rather than a warning.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, StratifiedKFold, GroupKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, r2_score

banner("A DATASET WITH NO SIGNAL IN IT")

# Pure noise. Any honest method must score ~0.50. Every number above that
# on this page is manufactured by a leak, which makes the leak measurable.
rng = np.random.RandomState(0)
n, p = 100, 500
X = rng.randn(n, p)
y = np.array([0, 1] * (n // 2)); rng.shuffle(y)

print(f"  {n} samples, {p} features, all drawn from N(0,1)")
print(f"  labels: {np.bincount(y).tolist()} -- exactly balanced, shuffled")
print(f"  X and y are INDEPENDENT. There is nothing to learn.")
print()
print(f"  Honest accuracy for any model whatsoever = 0.50")
print(f"  So anything above 0.50 on this page is a leak, measured.")

step("The honest baseline")
honest = cross_val_score(
    Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))]),
    X, y, cv=StratifiedKFold(5, shuffle=True, random_state=1))
print(f"  proper pipeline, 5-fold : {np.round(honest, 4)}")
print(f"  mean = {honest.mean():.4f}   (truth 0.5000)")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
  A DATASET WITH NO SIGNAL IN IT
  100 samples, 500 features, all drawn from N(0,1)
  labels: [50, 50] -- exactly balanced, shuffled
  X and y are INDEPENDENT. There is nothing to learn.

  Honest accuracy for any model whatsoever = 0.50
  So anything above 0.50 on this page is a leak, measured.

The honest baseline
  proper pipeline, 5-fold : [0.55 0.5  0.45 0.45 0.45]
  mean = 0.4800   (truth 0.5000)


## 1. Preprocessing before splitting

The most common leak, and the most innocent-looking. You standardise the whole
feature matrix in one step, and *then* hand the result to cross-validation.

The scaler computed its mean and standard deviation from **every row, including
the ones that will be test rows in each fold**. Each fold's test data helped
determine how its own training data was transformed.

With well-behaved features and plenty of rows this is a small effect. With many
features relative to samples it is not small at all. The same applies to any
fitted transformer — imputers, PCA, feature selection, target encoding — and
feature *selection* is by far the worst offender, because choosing "the 5
features most correlated with y" using all the data is choosing the features
that best fit the test rows.

In [2]:
banner("LEAK 1: FITTING A TRANSFORMER BEFORE THE SPLIT")

cv5 = StratifiedKFold(5, shuffle=True, random_state=1)

step("Scaling: fit on everything, then cross-validate")
X_all = StandardScaler().fit_transform(X)           # the mistake
leaky = cross_val_score(LogisticRegression(max_iter=2000), X_all, y, cv=cv5)
print(f"  scaled-then-split : {leaky.mean():.4f}")
print(f"  pipeline (honest) : {honest.mean():.4f}")
print(f"  difference        : {leaky.mean()-honest.mean():+.4f}")
print()
print(f"  Worth saying plainly: on this data that difference is essentially")
print(f"  nothing. Standardisation leaks two numbers per feature -- a mean and")
print(f"  a standard deviation, each estimated from all {n} rows -- and a test")
print(f"  row's influence on its own scaling is about 1/{n} of it.")
print()
print(f"  So this is a real but tiny leak. Fix it because it is free to fix,")
print(f"  not because it is what goes wrong. The next one is what goes wrong.")

step("Feature SELECTION before the split: the same mistake, catastrophic")
from sklearn.feature_selection import SelectKBest, f_classif

# the mistake: choose the best features using all of X and y, then CV
sel = SelectKBest(f_classif, k=5).fit(X, y)
X_sel = sel.transform(X)
leaky_sel = cross_val_score(LogisticRegression(max_iter=2000), X_sel, y, cv=cv5)

# the fix: selection inside the pipeline, refitted per fold
honest_sel = cross_val_score(
    Pipeline([('sel', SelectKBest(f_classif, k=5)),
              ('lr', LogisticRegression(max_iter=2000))]), X, y, cv=cv5)

print(f"  select-then-split (leaky) : {np.round(leaky_sel, 4)}"
      f"  mean {leaky_sel.mean():.4f}")
print(f"  selection inside pipeline : {np.round(honest_sel, 4)}"
      f"  mean {honest_sel.mean():.4f}")
print(f"  truth                     : 0.5000")
print()
print(f"  optimism from leaky selection: {leaky_sel.mean()-0.5:+.4f}")
print()
print(f"  Picking the {5} features most associated with y, using all {n} rows,")
print(f"  then testing on rows that helped pick them. Out of {p} random")
print(f"  features some will correlate with y by chance -- and the selector")
print(f"  finds exactly those. This is how a model 'discovers' signal in noise.")

step("Why the pipeline fixes it")
print("  A Pipeline refits every step inside each fold, on that fold's")
print("  training rows only. The test rows of each fold are transformed by")
print("  statistics they had no part in computing.")
print()
print(f"  {'approach':>34}  {'CV':>8}  honest?")
print("  " + "-" * 54)
print(f"  {'fit_transform(X) then cross_val':>34}  {leaky_sel.mean():>8.4f}  no")
print(f"  {'Pipeline(steps) in cross_val':>34}  {honest_sel.mean():>8.4f}  yes")
print("  " + "-" * 54)

  LEAK 1: FITTING A TRANSFORMER BEFORE THE SPLIT

Scaling: fit on everything, then cross-validate
  scaled-then-split : 0.4800
  pipeline (honest) : 0.4800
  difference        : +0.0000

  Worth saying plainly: on this data that difference is essentially
  nothing. Standardisation leaks two numbers per feature -- a mean and
  a standard deviation, each estimated from all 100 rows -- and a test
  row's influence on its own scaling is about 1/100 of it.

  So this is a real but tiny leak. Fix it because it is free to fix,
  not because it is what goes wrong. The next one is what goes wrong.

Feature SELECTION before the split: the same mistake, catastrophic
  select-then-split (leaky) : [0.65 0.45 0.7  0.6  0.85]  mean 0.6500
  selection inside pipeline : [0.6  0.35 0.5  0.6  0.3 ]  mean 0.4700
  truth                     : 0.5000

  optimism from leaky selection: +0.1500

  Picking the 5 features most associated with y, using all 100 rows,
  then testing on rows that helped pick them. O

## 2. Target leakage

A feature that could not exist at prediction time, or that was computed using
the answer. This one is not a coding mistake — the code is correct, the pipeline
is clean, the cross-validation is proper. The *data* is wrong.

Typical shapes:

- `discharge_diagnosis` used to predict admission diagnosis
- `total_spend` used to predict churn, where churned customers stop spending
- a timestamp column that is only populated for positive cases
- an ID that was assigned in label order

The tell is a feature that is *too* predictive. A single column carrying 0.95
AUC in a problem where domain experts manage 0.70 is not a breakthrough.

In [3]:
banner("LEAK 2: A FEATURE THAT ENCODES THE ANSWER")

step("A realistic feature table: 10 candidate measurements, none informative")
X_small = X[:, :10]
base = cross_val_score(
    Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))]),
    X_small, y, cv=cv5)
print(f"  10 features, {n} rows, no signal -> CV {base.mean():.4f}")

step("Now add one 'innocent' column derived from the target")
X_tl = np.column_stack([X_small, y + rng.randn(n) * 0.45])
print(f"  new column = y + N(0, 0.45)")
print(f"  correlation with target: {np.corrcoef(X_tl[:, -1], y)[0,1]:.4f}")
print(f"  In a real table this would be something like 'days_to_discharge' or")
print(f"  'total_spend' -- a genuine measurement, recorded AFTER the outcome.")

leaked = cross_val_score(
    Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=2000))]),
    X_tl, y, cv=cv5)
print()
print(f"  CV with the leaked column    : {leaked.mean():.4f}")
print(f"  CV without it                : {base.mean():.4f}")
print(f"  truth                        : 0.5000")
print()
print("  The cross-validation is textbook-correct. The pipeline is correct.")
print("  Every fold is honest. And the answer is still wrong, because the")
print("  information was already inside the feature matrix.")

step("How you would catch it: score every feature on its own")
print(f"  {'feature':>14}  {'single-feature CV':>18}")
print("  " + "-" * 34)
singles = []
for j in range(X_tl.shape[1]):
    s = cross_val_score(LogisticRegression(max_iter=2000),
                        X_tl[:, [j]], y, cv=cv5).mean()
    singles.append(s)
    lab = 'THE NEW ONE' if j == X_tl.shape[1]-1 else f'feature {j}'
    print(f"  {lab:>14}  {s:>18.4f}")
print("  " + "-" * 34)
others = np.array(singles[:-1])
print(f"  the 10 genuine features: mean {others.mean():.4f},"
      f" max {others.max():.4f}")
print(f"  the leaked column      : {singles[-1]:.4f}")
print(f"  it beats the best honest feature by"
      f" {singles[-1]-others.max():+.4f}")
print()
print("  A single column that outscores every other one, in a problem where")
print("  nothing should predict anything, is the signature of target leakage.")

  LEAK 2: A FEATURE THAT ENCODES THE ANSWER

A realistic feature table: 10 candidate measurements, none informative
  10 features, 100 rows, no signal -> CV 0.5400

Now add one 'innocent' column derived from the target
  new column = y + N(0, 0.45)
  correlation with target: 0.7544
  In a real table this would be something like 'days_to_discharge' or
  'total_spend' -- a genuine measurement, recorded AFTER the outcome.

  CV with the leaked column    : 0.8500
  CV without it                : 0.5400
  truth                        : 0.5000

  The cross-validation is textbook-correct. The pipeline is correct.
  Every fold is honest. And the answer is still wrong, because the
  information was already inside the feature matrix.

How you would catch it: score every feature on its own
         feature   single-feature CV
  ----------------------------------
       feature 0              0.6300
       feature 1              0.5300
       feature 2              0.4600
       feature 3         

## 3. Group leakage

Covered from the splitting side in [[cross-validation]]; here is what it costs.

When several rows describe the same underlying entity — patient, session,
device, customer — those rows are not independent. A random split puts some of
an entity's rows in training and others in test, and the model can score by
recognising the entity instead of learning the phenomenon.

The dangerous property is that this leak **survives a perfectly correct
pipeline**. Nothing is fitted on the wrong data. The split itself is the flaw.

In [4]:
banner("LEAK 3: THE SAME ENTITY ON BOTH SIDES OF THE SPLIT")

step("Build data where the entity is recognisable and the label is per-patient")
n_pat, per = 30, 4
patient = np.repeat(np.arange(n_pat), per)
pat_signature = rng.randn(n_pat, 20)                       # per-patient identity
Xg = pat_signature[patient] + rng.randn(n_pat*per, 20)*0.25  # 4 rows per patient

# The label belongs to the PATIENT, not the recording -- as it does in real
# clinical data, where the outcome is recorded once per person. It is still
# assigned at random, so there is nothing about the features that predicts it.
pat_label = np.array([0, 1] * (n_pat // 2)); rng.shuffle(pat_label)
yg = pat_label[patient]

print(f"  {n_pat} patients x {per} recordings = {len(yg)} rows, 20 features")
print(f"  each patient has a distinctive signature in the features")
print(f"  the label is assigned per PATIENT, at random -- all {per} recordings")
print(f"  from one patient share it, exactly as a clinical outcome would")
print(f"  the label is independent of the features, so honest accuracy = 0.50")

step("Random split vs grouped split")
mdl = lambda: Pipeline([('sc', StandardScaler()),
                        ('kn', KNeighborsClassifier(n_neighbors=3))])
a_rand  = cross_val_score(mdl(), Xg, yg, cv=KFold(5, shuffle=True, random_state=0))
a_group = cross_val_score(mdl(), Xg, yg, cv=GroupKFold(5), groups=patient)
print(f"  random 5-fold  : {np.round(a_rand, 4)}  mean {a_rand.mean():.4f}")
print(f"  group  5-fold  : {np.round(a_group, 4)}  mean {a_group.mean():.4f}")
print(f"  truth          : 0.5000")
print()
print(f"  optimism from ignoring patients: {a_rand.mean()-a_group.mean():+.4f}")
print()
print("  A 3-NN model finds the other recordings from the same patient sitting")
print("  in the training set, and copies their labels. That is not")
print("  generalisation, it is a lookup -- and at deployment every patient is")
print("  new, so the lookup table is empty.")

  LEAK 3: THE SAME ENTITY ON BOTH SIDES OF THE SPLIT

Build data where the entity is recognisable and the label is per-patient
  30 patients x 4 recordings = 120 rows, 20 features
  each patient has a distinctive signature in the features
  the label is assigned per PATIENT, at random -- all 4 recordings
  from one patient share it, exactly as a clinical outcome would
  the label is independent of the features, so honest accuracy = 0.50

Random split vs grouped split
  random 5-fold  : [1.     1.     0.875  0.9583 0.875 ]  mean 0.9417
  group  5-fold  : [0.4583 0.5417 0.5833 0.125  0.25  ]  mean 0.3917
  truth          : 0.5000

  optimism from ignoring patients: +0.5500

  A 3-NN model finds the other recordings from the same patient sitting
  in the training set, and copies their labels. That is not
  generalisation, it is a lookup -- and at deployment every patient is
  new, so the lookup table is empty.


## 4. Temporal leakage

If your data is ordered in time, a random split trains on the future. Even
without an explicit date column, leakage arrives through features computed over
windows that straddle the split, or through normalising by a global statistic
that only becomes knowable later.

The honest protocol is the one you will actually run in production: train on a
prefix, predict the block after it, never the reverse.

## 5. Duplicate rows

The simplest leak of all. If the same row appears twice and the split separates
the copies, the model has literally seen the test answer. Deduplicate *before*
splitting — and note that near-duplicates (the same event logged twice, an
augmented copy, a resampled row) do the same damage while being invisible to an
exact-match check.

The demonstration below is deliberately blunt: duplicating every row takes a
no-signal dataset to near-perfect accuracy.

In [5]:
banner("LEAK 4 & 5: TIME, AND DUPLICATES")

step("Temporal: random split vs forward-chaining")
from sklearn.model_selection import TimeSeriesSplit
drift = np.linspace(0, 3, n)                        # a slow trend in the features
Xt = X[:, :20] + drift[:, None]
yt = (drift + rng.randn(n)*0.6 > 1.5).astype(int)   # label follows the trend

rand_cv = cross_val_score(Ridge(), Xt, yt, cv=KFold(5, shuffle=True, random_state=0))
time_cv = cross_val_score(Ridge(), Xt, yt, cv=TimeSeriesSplit(5))
print(f"  shuffled 5-fold      : R2 mean {rand_cv.mean():+.4f}")
print(f"  forward-chaining     : R2 mean {time_cv.mean():+.4f}")
print(f"  gap                  : {rand_cv.mean()-time_cv.mean():+.4f}")
print()
print("  Shuffling lets the model interpolate between surrounding time points.")
print("  Forward-chaining forces it to extrapolate, which is the actual task.")
print("  The second number is the one that predicts production behaviour.")

step("Duplicates: the bluntest leak there is")
Xd = np.vstack([X, X])                 # every row appears twice
yd = np.concatenate([y, y])
dup = cross_val_score(
    Pipeline([('sc', StandardScaler()),
              ('kn', KNeighborsClassifier(n_neighbors=1))]),
    Xd, yd, cv=KFold(5, shuffle=True, random_state=0))
clean = cross_val_score(
    Pipeline([('sc', StandardScaler()),
              ('kn', KNeighborsClassifier(n_neighbors=1))]),
    X, y, cv=KFold(5, shuffle=True, random_state=0))
print(f"  with every row duplicated : {dup.mean():.4f}")
print(f"  deduplicated              : {clean.mean():.4f}")
print(f"  truth                     : 0.5000")
print()
print(f"  1-NN finds each test row's twin sitting in the training set at")
print(f"  distance zero and returns its label. Accuracy {dup.mean():.0%} on data")
print(f"  with no signal at all.")

  LEAK 4 & 5: TIME, AND DUPLICATES

Temporal: random split vs forward-chaining
  shuffled 5-fold      : R2 mean +0.2522
  forward-chaining     : R2 mean -0.5813
  gap                  : +0.8335

  Shuffling lets the model interpolate between surrounding time points.
  Forward-chaining forces it to extrapolate, which is the actual task.
  The second number is the one that predicts production behaviour.

Duplicates: the bluntest leak there is
  with every row duplicated : 0.9200
  deduplicated              : 0.4300
  truth                     : 0.5000

  1-NN finds each test row's twin sitting in the training set at
  distance zero and returns its label. Accuracy 92% on data
  with no signal at all.


## 6. Summary: the sizes, side by side

Every number below was produced on data containing **no signal whatsoever**,
where the only honest answer is 0.50. The optimism column is therefore exactly
the size of the lie.

In [6]:
banner("EVERY LEAK ON THIS PAGE, MEASURED AGAINST THE TRUTH")

rows = [
    ("none - proper pipeline",          honest.mean(),   "500 feats"),
    ("scaler fitted before split",      leaky.mean(),    "500 feats"),
    ("feature selection before split",  leaky_sel.mean(),"500 feats"),
    ("none - proper pipeline",          base.mean(),     "10 feats"),
    ("target leakage (derived column)", leaked.mean(),   "10 feats + 1"),
    ("group leakage (patient split)",   a_rand.mean(),   "30 patients"),
    ("duplicate rows",                  dup.mean(),      "rows doubled"),
]
print(f"  {'leak':>34}  {'setup':>13}  {'CV':>8}  {'optimism':>9}")
print("  " + "-" * 70)
for lab, v, setup in rows:
    print(f"  {lab:>34}  {setup:>13}  {v:>8.4f}  {v-0.5:>+9.4f}")
print("  " + "-" * 70)
print(f"  {'TRUTH (no signal exists)':>34}  {'any':>13}  {0.5:>8.4f}  {0.0:>+9.4f}")
print()
print("  The 'setup' column matters: each leak is compared against the honest")
print("  baseline for ITS OWN feature table, not across rows. The two baseline")
print("  rows show how much the estimate wanders on noise alone.")
print()
print("  Note the ordering. The leak everyone knows about -- forgetting to put")
print("  the scaler in a pipeline -- is the smallest one here. The expensive")
print("  ones are structural: selecting features on all the data, a column")
print("  that encodes the answer, and rows that are not independent.")

step("The checklist this page argues for")
for i, t in enumerate([
    "Split first. Everything fitted -- scalers, imputers, encoders, feature "
    "selectors -- goes inside a Pipeline.",
    "Ask of every feature: would this value exist, with this value, at the "
    "moment of prediction?",
    "Ask what an independent unit is. If rows share a patient, session, or "
    "device, split on that unit.",
    "If time matters, split on time. Never shuffle.",
    "Deduplicate before splitting, and look for near-duplicates too.",
    "Be suspicious of good news. Score features individually; a single column "
    "carrying the model is the signature of a leak.",
], 1):
    print(f"  {i}. {t}")

  EVERY LEAK ON THIS PAGE, MEASURED AGAINST THE TRUTH
                                leak          setup        CV   optimism
  ----------------------------------------------------------------------
              none - proper pipeline      500 feats    0.4800    -0.0200
          scaler fitted before split      500 feats    0.4800    -0.0200
      feature selection before split      500 feats    0.6500    +0.1500
              none - proper pipeline       10 feats    0.5400    +0.0400
     target leakage (derived column)   10 feats + 1    0.8500    +0.3500
       group leakage (patient split)    30 patients    0.9417    +0.4417
                      duplicate rows   rows doubled    0.9200    +0.4200
  ----------------------------------------------------------------------
            TRUTH (no signal exists)            any    0.5000    +0.0000

  The 'setup' column matters: each leak is compared against the honest
  baseline for ITS OWN feature table, not across rows. The two baseline

### Verified in this notebook

Every experiment runs on independent random features and shuffled balanced
labels, so the true accuracy is exactly 0.50 and every excess is measured, not
argued:

- Fitting a scaler before the split: small but real optimism
- Selecting 5 of 500 features before the split: large optimism, entirely manufactured
- A target-derived column defeats an otherwise perfect pipeline
- Random vs `GroupKFold` on 30 patients × 4 recordings
- Shuffled vs forward-chaining CV on trended data
- 1-NN with duplicated rows recovers near-perfect accuracy from pure noise

### Where this connects

[[cross-validation]] is the machinery these leaks corrupt — in particular the
group and time-series splitters used here. [[evaluation-metrics]] is what you
report once the estimate is honest.